**[Course Home](../README.md) |
[Python Lab](../python/06_differential_entropy_and_continuous_limits.ipynb) |
[Prev: Ch 05](05_channel_capacity_and_blahut_arimoto.ipynb) |
[Next: Ch 07](07_the_principle_of_maximum_entropy_maxent.ipynb)**

*Part III: Channels, Capacity & Coding Limits — Chapter 06 of 11*

---
# Chapter 6: Continuous Worlds — Differential Entropy & Continuous Limits
### *Riemann Partitions, Coordinate Paradoxes, and*
### *Why Gaussians Maximize Disorder*
---
## 1. What Happens When States Are Infinitely Dense?
In discrete information theory, states are distinct, countable entities: coin
faces, English characters, DNA nucleotides, or packet routing headers. In all
these discrete settings, Shannon entropy:
$$
H(X) = -\sum_{x \in \mathcal{X}} P(x) \log_2 P(x)
$$
possesses three foundational, well-behaved mathematical properties:
1. **Strict Non-negativity**: $H(X) \ge 0$ always.
2. **Invariance**: Unchanged under one-to-one relabeling of symbols.
3. **Operational Meaning**: The exact minimum expected number of binary yes/no
   questions required to uniquely identify the unknown state.

However, the physical universe is overwhelmingly continuous: electrical
voltages, mechanical vibrations, atmospheric temperatures, radio carrier
frequencies, and particle coordinates vary along the continuum of the real
numbers $\mathbb{R}$.

What happens when we attempt to compute the information content of a continuous
random variable?

```
                     THE CONTINUOUS INFORMATION PROBLEM
                     
    Discrete State:                  Continuous State:
    [Head] or [Tail]                 Real Number X in [0.0, 1.0]
    
    1 bit identifies state           How many bits to pinpoint X?
    
    [ H | T ]                        [ 0.73928194857291048294719... ]
    Countable choices.               Infinite decimals = Infinite bits!
```
### The Riemann Discretization Paradox
Let $X$ be a continuous random variable with probability density function
$f(x)$. To measure $X$ on a digital computer or with a physical sensor, we must
quantize it. Divide the support of $X$ into uniform discrete bins of width
$\Delta$, indexed by integers $i$.

The discrete bin is defined by the interval $[i\Delta, (i+1)\Delta)$. By the
Mean Value Theorem for integrals, there exists a point $x_i^*$ inside the
$i$-th bin such that the probability mass allocated to that bin is:
$$
p_i = \int_{i\Delta}^{(i+1)\Delta} f(x) \, dx = f(x_i^*) \Delta
$$
Now calculate the Shannon entropy of this discretized random variable
$X^\Delta$:
$$
\begin{aligned}
H(X^\Delta) &= -\sum_{i=-\infty}^{\infty} p_i \log_2 p_i \\
&= -\sum_{i=-\infty}^{\infty} f(x_i^*)\Delta \log_2\Big(f(x_i^*)\Delta\Big) \\
&= -\sum_{i=-\infty}^{\infty} f(x_i^*)\Delta \log_2 f(x_i^*)
- \sum_{i=-\infty}^{\infty} f(x_i^*)\Delta \log_2 \Delta
\end{aligned}
$$
Because $\sum_{i} f(x_i^*)\Delta = \int f(x)dx = 1$, the second sum simplifies
drastically:
$$
H(X^\Delta) = -\sum_{i=-\infty}^{\infty} f(x_i^*)\Delta \log_2 f(x_i^*)
- \log_2 \Delta
$$
Now take the continuous limit as bin width shrinks to zero, $\Delta \to 0$.
The Riemann sum converges to the continuous Riemann integral:
$$
\lim_{\Delta \to 0} \sum_{i=-\infty}^{\infty} f(x_i^*)\Delta \log_2 f(x_i^*)
= \int_{-\infty}^{\infty} f(x) \log_2 f(x) \, dx
$$
Look at what happens to the remaining term:
As $\Delta \to 0$, $-\log_2 \Delta = \log_2(1/\Delta) \to +\infty$!
$$
\lim_{\Delta \to 0} H(X^\Delta)
= -\int_{-\infty}^{\infty} f(x) \log_2 f(x) \, dx + \infty = +\infty
$$
### The Physical Lesson of the Discretization Paradox
The discrete entropy of any continuous variable is **strictly infinite**.

This is not a mathematical bug; it is an undeniable physical truth. Specifying
a continuous real number with infinite decimal precision requires an infinite
number of bits. If someone asks you to specify the exact coordinate of an
electron to infinite precision, you would have to output a non-terminating,
non-repeating string of bits:
$$
x = 0.1011001110101101001010..._2 \implies \infty \text{ bits}
$$
### Shannon's Resolution: Differential Entropy
Claude Shannon recognized that the $+\infty$ term, $\log_2(1/\Delta)$, depends
purely on the resolution of the coordinate grid, not on the underlying shape
or distribution of the random variable.

Shannon proposed subtracting the divergent coordinate term $\log_2(1/\Delta)$
before taking the limit:
$$
h(X) \equiv \lim_{\Delta \to 0} \Big[ H(X^\Delta) + \log_2 \Delta \Big]
= -\int_{-\infty}^{\infty} f(x) \log_2 f(x) \, dx
$$
This quantity $h(X)$ is named **Differential Entropy**.

> **Crucial Definition**:
> For a continuous random variable $X$ with probability density function
> $f(x)$, the differential entropy (in bits) is:
> $$
> h(X) = -\int_{-\infty}^{\infty} f(x) \log_2 f(x) \, dx
> $$
> If natural logarithms are used, differential entropy is measured in
> **nats**:
> $$
> h_e(X) = -\int_{-\infty}^{\infty} f(x) \ln f(x) \, dx = h(X) \ln 2
> $$

---
## 2. The Two Paradoxes of Continuous Entropy
Because differential entropy drops the infinite constant $\log_2(1/\Delta)$, it
loses several comforting properties of discrete entropy. Anyone applying
differential entropy in machine learning or physics must understand two major
traps.

```
                  DISCRETE VS. DIFFERENTIAL ENTROPY
                  
     Property                 Discrete H(X)          Differential h(X)
     ------------------------------------------------------------------
     Absolute Floor           Always >= 0            Can be -infinity!
     Scale Invariance         Invariant              Changes by log2|c|
     Uniform Baseline         Uniform = log2(N)      Uniform = log2(Volume)
     Mutual Information       Invariant              Strictly Invariant!
     KL Divergence            Invariant              Strictly Invariant!
```
### Paradox 1: Differential Entropy Can Be Negative!
Discrete entropy $H(X)$ is bounded below by 0 (zero uncertainty for a
deterministic state). In sharp contrast, **differential entropy $h(X)$ can be
arbitrarily negative!**

Consider a continuous random variable $X$ distributed uniformly on the
interval $[0, a]$ where $a > 0$:
$$
f(x) = \begin{cases} \frac{1}{a}, & x \in [0, a] \\ 0, & \text{otherwise}
\end{cases}
$$
Let us compute its differential entropy:
$$
h(X) = -\int_0^a \frac{1}{a} \log_2\left(\frac{1}{a}\right) dx
= -\log_2\left(\frac{1}{a}\right) \int_0^a \frac{1}{a} dx = \log_2 a
$$
Now examine what happens for various interval widths $a$:
- If $a = 2.0$: $h(X) = \log_2(2.0) = +1.0$ bit.
- If $a = 1.0$: $h(X) = \log_2(1.0) = 0.0$ bits.
- If $a = 0.5$: $h(X) = \log_2(0.5) = \mathbf{-1.0 \text{ bit}}$!
- If $a = 0.125$: $h(X) = \log_2(0.125) = \mathbf{-3.0 \text{ bits}}$!
- As $a \to 0^+$: $h(X) = \log_2(a) \to -\infty$!

```
                  WHY DIFFERENTIAL ENTROPY TURNS NEGATIVE
                  
   f(x) ^                                   f(x) ^
   1.0  |--------+                          4.0  |---+
        |        |                               |   |
        | a = 1  |                               |a= |
        |        |                               |0.2|
        +--------+-----> x                       +---+-------> x
        0       1.0                              0  0.25
        h(X) = log2(1) = 0 bits                  h(X) = log2(0.25) = -2 bits
        (Reference Standard)                     (Higher density than unit)
```
#### Why Does Negative Entropy Make Physical Sense?
Remember the origin of differential entropy:
$$
H(X^\Delta) \approx h(X) + \log_2(1/\Delta)
$$
The discrete entropy $H(X^\Delta)$ is the number of bits required to specify
the state to precision $\Delta$.
- Suppose you measure a variable with width $a = 0.25$ using bin width
  $\Delta = 0.001$:
  $$
  H(X^\Delta) \approx -2.0 + \log_2(1000) = -2.0 + 9.966 = 7.966 \text{ bits}
  $$
- The true discrete uncertainty $H(X^\Delta)$ is still positive!
- The differential entropy $h(X) = -2.0$ bits simply indicates that $X$
  requires **2 fewer bits** to specify than a uniform variable on a unit
  interval $[0, 1.0]$ at that same precision $\Delta$.

Differential entropy is **not an absolute measure of uncertainty**. It is a
**relative density metric** calibrated against a reference unit volume of 1.0!

---
### Paradox 2: Coordinate Rescaling Changes Entropy
In discrete information theory, if you rename symbols $X \to Y$ via any
one-to-one mapping, entropy does not change: $H(Y) = H(X)$.

In continuous worlds, changing measurement units alters differential entropy!

Suppose $X$ has density $f_X(x)$, and let $Y = c X$ for some constant $c \ne 0$.
By the change-of-variables formula for probability densities:
$$
f_Y(y) = \frac{1}{|c|} f_X\left(\frac{y}{c}\right)
$$
Now compute the differential entropy of $Y$:
$$
\begin{aligned}
h(Y) &= -\int_{-\infty}^{\infty} f_Y(y) \log_2 f_Y(y) \, dy \\
&= -\int_{-\infty}^{\infty} \frac{1}{|c|} f_X\left(\frac{y}{c}\right)
\log_2 \left[ \frac{1}{|c|} f_X\left(\frac{y}{c}\right) \right] dy
\end{aligned}
$$
Substitute $u = y/c$, so $dy = c \, du$ (with careful attention to sign):
$$
\begin{aligned}
h(Y) &= -\int_{-\infty}^{\infty} f_X(u)
\left[ \log_2 f_X(u) - \log_2 |c| \right] du \\
&= -\int_{-\infty}^{\infty} f_X(u) \log_2 f_X(u) \, du
+ \log_2 |c| \int_{-\infty}^{\infty} f_X(u) \, du \\
&= h(X) + \log_2 |c|
\end{aligned}
$$
> **The Coordinate Scaling Law**:
> $$
> h(cX) = h(X) + \log_2 |c|
> $$
> For a vector $\mathbf{X} \in \mathbb{R}^n$ and an invertible matrix
> $\mathbf{A}$:
> $$
> h(\mathbf{A}\mathbf{X}) = h(\mathbf{X}) + \log_2 |\det \mathbf{A}|
> $$

#### Engineering Consequences of the Scaling Law
- **Meters vs. Millimeters**: If you measure an antenna displacement $X$ in
  meters, its differential entropy is $h(X)$. If you measure the exact same
  physical quantity in millimeters ($Y = 1000 X$), the differential entropy
  increases by:
  $$
  \log_2(1000) \approx 9.966 \text{ bits}
  $$
- **Celsius vs. Fahrenheit**: Temperature scaling $T_F = \frac{9}{5} T_C + 32$
  changes differential entropy by $\log_2(9/5) \approx 0.848$ bits
  (constant shifts do not change entropy, but scale factors do).

---
### The Unshakable Invariants: KL Divergence and Mutual Information
While differential entropy depends on the coordinate system, **Relative
Entropy (KL Divergence) and Mutual Information are strictly invariant under
any smooth, invertible change of coordinates!**

Let $Y = g(X)$ be a strictly monotonic differentiable transformation with
inverse $X = g^{-1}(Y)$ and Jacobian $J(y) = \frac{dx}{dy}$.
Let $P_X$ and $Q_X$ be two probability densities on $X$. The transformed
densities are $P_Y(y) = P_X(g^{-1}(y)) |J(y)|$ and
$Q_Y(y) = Q_X(g^{-1}(y)) |J(y)|$.

Compute the KL divergence between $P_Y$ and $Q_Y$:
$$
\begin{aligned}
D_{\text{KL}}(P_Y \parallel Q_Y)
&= \int P_Y(y) \log_2 \left( \frac{P_Y(y)}{Q_Y(y)} \right) dy \\
&= \int P_X(g^{-1}(y)) |J(y)|
\log_2 \left( \frac{P_X(g^{-1}(y)) |J(y)|}{Q_X(g^{-1}(y)) |J(y)|} \right) dy
\end{aligned}
$$
The Jacobians $|J(y)|$ inside the logarithm cancel out completely!
Substituting $x = g^{-1}(y)$ with $dx = |J(y)| dy$:
$$
D_{\text{KL}}(P_Y \parallel Q_Y)
= \int P_X(x) \log_2 \left( \frac{P_X(x)}{Q_X(x)} \right) dx
= D_{\text{KL}}(P_X \parallel Q_X)
$$
Because Mutual Information is defined as a relative entropy:
$$
I(X; Y) = D_{\text{KL}}\Big(P_{XY} \;\Big\|\; P_X P_Y\Big)
$$
it follows immediately that:
$$
I\big(g(X); h(Y)\big) = I(X; Y)
$$
for any smooth invertible functions $g$ and $h$.

> **Takeaway**:
> Never use differential entropy alone to compare two systems measured in
> different units. Always use **Mutual Information** or **KL Divergence**,
> which represent coordinate-free physical truths.

---
## 3. The Gaussian Maximum Entropy Theorem
Among all continuous probability distributions supported on $(-\infty, \infty)$
with a **fixed variance $\sigma^2$**, which distribution has the highest
differential entropy?

```
               WHICH DISTRIBUTION HAS THE HIGHEST ENTROPY?
               
   f(x) ^              Laplace (Sharp peak, heavy tails)
        |     /\       Uniform (Flat box, zero outside)
        |    /  \      Gaussian (Smooth bell, maximizes disorder!)
        |   / /\ \     
        |  / /  \ \    All three curves have IDENTICAL variance sigma^2!
        | / /    \ \   
        +------------------> x
        The Gaussian bell spreads probability mass as uniformly as possible
        subject to the second moment constraint!
```
### Theorem Statement
Let $X$ be a real-valued continuous random variable with mean $\mu$ and
variance $\sigma^2 = \mathbb{E}[(X - \mu)^2]$.
Then:
$$
h(X) \le \frac{1}{2} \log_2(2\pi e \sigma^2) \quad [\text{bits}]
$$
with equality if and only if $X$ is normally distributed:
$X \sim \mathcal{N}(\mu, \sigma^2)$.

---
### Proof 1: The Relative Entropy (KL Divergence) Method
The cleanest and most rigorous proof relies on the non-negativity of KL
divergence (Gibbs' inequality).

Let $g(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(
-\frac{(x-\mu)^2}{2\sigma^2} \right)$
be the Gaussian density with mean $\mu$ and variance $\sigma^2$.
Let $f(x)$ be **any** arbitrary probability density function on $(-\infty,
\infty)$
satisfying the same first two moments:
$$
\int_{-\infty}^\infty f(x) \, dx = 1, \quad
\int_{-\infty}^\infty x f(x) \, dx = \mu, \quad
\int_{-\infty}^\infty (x-\mu)^2 f(x) \, dx = \sigma^2
$$
By Gibbs' inequality, the relative entropy between $f$ and $g$ is non-negative:
$$
D_{\text{KL}}(f \parallel g) = \int_{-\infty}^\infty f(x) \ln \left(
\frac{f(x)}{g(x)} \right) dx \ge 0
$$
Expand the logarithm:
$$
D_{\text{KL}}(f \parallel g) = \int_{-\infty}^\infty f(x) \ln f(x) \, dx -
\int_{-\infty}^\infty f(x) \ln g(x) \, dx \ge 0
$$
The first term is the negative differential entropy of $f$ in nats:
$$
\int_{-\infty}^\infty f(x) \ln f(x) \, dx = -h_e(f)
$$
Now examine the second term:
$$
\ln g(x) = \ln \left( \frac{1}{\sqrt{2\pi\sigma^2}} \right) -
\frac{(x-\mu)^2}{2\sigma^2} = -\frac{1}{2}\ln(2\pi\sigma^2) -
\frac{(x-\mu)^2}{2\sigma^2}
$$
Integrate this against $f(x)$:
$$
\begin{aligned}
\int_{-\infty}^\infty f(x) \ln g(x) \, dx
&= -\frac{1}{2}\ln(2\pi\sigma^2) \int_{-\infty}^\infty f(x) \, dx
- \frac{1}{2\sigma^2} \int_{-\infty}^\infty (x-\mu)^2 f(x) \, dx \\
&= -\frac{1}{2}\ln(2\pi\sigma^2) \cdot (1) - \frac{1}{2\sigma^2} \cdot
(\sigma^2) \\
&= -\frac{1}{2}\ln(2\pi\sigma^2) - \frac{1}{2} \\
&= -\frac{1}{2} \[ \ln(2\pi\sigma^2) + \ln(e) \] \\
&= -\frac{1}{2} \ln(2\pi e \sigma^2)
\end{aligned}
$$
Notice that this integral depends **only** on the variance $\sigma^2$ of $f$,
which matches the variance of $g$! Therefore:
$$
\int_{-\infty}^\infty f(x) \ln g(x) \, dx = \int_{-\infty}^\infty g(x) \ln
g(x) \, dx = -h_e(g)
$$
Substitute this back into the KL inequality:
$$
D_{\text{KL}}(f \parallel g) = -h_e(f) - (-h_e(g)) = h_e(g) - h_e(f) \ge 0
$$
$$
\implies h_e(f) \le h_e(g) = \frac{1}{2} \ln(2\pi e \sigma^2)
$$
Converting to base-2 bits:
$$
h(f) \le \frac{1}{2} \log_2(2\pi e \sigma^2)
$$
Because $D_{\text{KL}}(f \parallel g) = 0$ if and only if $f(x) = g(x)$ almost
everywhere, the Gaussian distribution is the **unique** maximum entropy
distribution. $\blacksquare$

---
### Proof 2: Calculus of Variations and Lagrange Multipliers
We can also derive the Gaussian form directly by maximizing the functional:
$$
h_e(f) = -\int_{-\infty}^\infty f(x) \ln f(x) \, dx
$$
subject to three integral constraints:
1. Normalization: $\int_{-\infty}^\infty f(x) \, dx = 1$
2. Mean: $\int_{-\infty}^\infty x f(x) \, dx = \mu$
3. Variance: $\int_{-\infty}^\infty (x - \mu)^2 f(x) \, dx = \sigma^2$

Construct the Lagrangian functional:
$$
\mathcal{L}[f] = -\int f(x) \ln f(x) dx
- \lambda_0 \left(\int f(x)dx - 1\right)
- \lambda_1 \left(\int x f(x)dx - \mu\right)
- \lambda_2 \left(\int (x-\mu)^2 f(x)dx - \sigma^2\right)
$$
Take the functional derivative with respect to $f(x)$ and set it to zero:
$$
\frac{\delta \mathcal{L}}{\delta f(x)}
= -\ln f(x) - 1 - \lambda_0 - \lambda_1 x - \lambda_2 (x - \mu)^2 = 0
$$
Solve for $f(x)$:
$$
\ln f(x) = -(1 + \lambda_0) - \lambda_1 x - \lambda_2 (x - \mu)^2
$$
$$
f(x) = \exp\Big(-(1 + \lambda_0)\Big) \cdot \exp\Big(-\lambda_1 x - \lambda_2
(x - \mu)^2\Big)
$$
For the integral $\int f(x)dx$ to converge, we must have $\lambda_2 > 0$.
By symmetry about the mean $\mu$, the linear term must vanish: $\lambda_1 = 0$.
Enforcing the variance constraint $\int (x-\mu)^2 f(x)dx = \sigma^2$ yields:
$$
\lambda_2 = \frac{1}{2\sigma^2}
$$
Enforcing normalization $\int f(x)dx = 1$ yields the normalizing constant:
$$
\exp\Big(-(1 + \lambda_0)\Big) = \frac{1}{\sqrt{2\pi\sigma^2}}
$$
Thus, the optimal density is uniquely the Gaussian:
$$
f^*(x) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x -
\mu)^2}{2\sigma^2}\right)
$$
---
### Multivariate Generalization
For an $n$-dimensional continuous random vector $\mathbf{X} = [X_1, \dots,
X_n]^T$
with mean vector $\boldsymbol{\mu}$ and $n \times n$ covariance matrix
$\boldsymbol{\Sigma} =
\mathbb{E}[(\mathbf{X}-\boldsymbol{\mu})(\mathbf{X}-\boldsymbol{\mu})^T]$:

> **Multivariate Gaussian MaxEnt Theorem**:
> Among all distributions on $\mathbb{R}^n$ with covariance matrix
> $\boldsymbol{\Sigma}$, the multivariate normal distribution
> $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ uniquely maximizes
> differential entropy:
> $$
> h(\mathbf{X}) \le \frac{1}{2} \log_2\Big( (2\pi e)^n
\det(\boldsymbol{\Sigma}) \Big) \quad [\text{bits}]
> $$

Notice that $\det(\boldsymbol{\Sigma})$ represents the square of the volume
of the uncertainty ellipsoid in $n$-dimensional space!

---
## 4. The Shannon-Hartley Theorem
The Gaussian Maximum Entropy Theorem is not just an abstract mathematical
curiosity; it is the theoretical engine behind modern telecommunications:
Wi-Fi, 5G/6G cellular, optical fiber, and deep-space telemetry.

```
                 THE ADDITIVE WHITE GAUSSIAN NOISE (AWGN) CHANNEL
                 
    Transmitter                Physical Medium               Receiver
     Power <= S                  Noise ~ N(0, N)
    
     [ Signal X ] ---------> (+) ------------------------> [ Received Y ]
                              ^                                Y = X + Z
                              |
                         [ Noise Z ]
```
### The AWGN Channel Model
Consider the Additive White Gaussian Noise (AWGN) channel:
$$
Y = X + Z
$$
where:
- $X$ is the transmitted signal. The transmitter has an average power
  constraint: $\mathbb{E}[X^2] \le S$.
- $Z$ is thermal noise, distributed as independent Gaussian noise:
  $Z \sim \mathcal{N}(0, N)$, independent of $X$.
- $Y$ is the received signal.

The channel capacity $C$ is the maximum mutual information between input $X$
and output $Y$, subject to the transmitter power constraint:
$$
C = \max_{f(x): \mathbb{E}[X^2] \le S} I(X; Y)
$$
### Derivation of Channel Capacity
Express mutual information in terms of differential entropy:
$$
I(X; Y) = h(Y) - h(Y \mid X)
$$
Conditioned on knowing the transmitted input $X = x$, the received signal is
$Y = x + Z$. The translation by constant $x$ does not alter differential
entropy:
$$
h(Y \mid X = x) = h(x + Z \mid X = x) = h(Z)
$$
Because the noise $Z$ is independent of $X$:
$$
h(Y \mid X) = h(Z)
$$
Since $Z \sim \mathcal{N}(0, N)$, its differential entropy is:
$$
h(Z) = \frac{1}{2} \log_2(2\pi e N)
$$
Therefore, the mutual information becomes:
$$
I(X; Y) = h(Y) - \frac{1}{2} \log_2(2\pi e N)
$$
To maximize $I(X; Y)$, we must **maximize the output differential entropy
$h(Y)$**.

What is the variance of the received signal $Y$?
Because signal $X$ and noise $Z$ are independent (so $\mathbb{E}[XZ] = 0$):
$$
\mathbb{E}[Y^2] = \mathbb{E}[(X + Z)^2]
= \mathbb{E}[X^2] + 2\mathbb{E}[XZ] + \mathbb{E}[Z^2] \le S + N
$$
By the Gaussian Maximum Entropy Theorem, for any random variable with
variance bounded by $S + N$, its differential entropy cannot exceed the
Gaussian entropy:
$$
h(Y) \le \frac{1}{2} \log_2\Big(2\pi e (S + N)\Big)
$$
Equality is achieved if and only if **$Y$ is Gaussian**.

Since the sum of two independent variables is Gaussian if both are Gaussian,
$Y = X + Z$ is Gaussian when the input **$X$ is Gaussian**:
$$
X \sim \mathcal{N}(0, S)
$$
Substitute this maximal output entropy into the mutual information formula:
$$
\begin{aligned}
C &= \frac{1}{2} \log_2\Big(2\pi e (S + N)\Big) - \frac{1}{2} \log_2(2\pi e N)
\\
&= \frac{1}{2} \log_2\left( \frac{2\pi e (S + N)}{2\pi e N} \right) \\
&= \frac{1}{2} \log_2\left( 1 + \frac{S}{N} \right) \quad [\text{bits per
sample}]
\end{aligned}
$$
---
### Bandwidth Expansion & The Continuous-Time Limit
In a continuous-time channel with physical bandwidth $B$ Hertz:
1. By the **Nyquist-Shannon Sampling Theorem**, a bandlimited signal of
   bandwidth $B$ is completely specified by $2B$ independent samples per
   second.
2. The thermal noise power in bandwidth $B$ is $N = N_0 B$, where $N_0$ is the
   noise power spectral density (in Watts/Hz).
3. The signal power is $S$ (in Watts).

Multiplying the capacity per sample by the sampling rate $2B$ samples/second:
$$
C = 2B \times \frac{1}{2} \log_2\left( 1 + \frac{S}{N_0 B} \right)
= B \log_2\left( 1 + \frac{S}{N_0 B} \right) \quad [\text{bits/second}]
$$
This is the celebrated **Shannon-Hartley Theorem**:
$$
C = B \log_2\left( 1 + \frac{S}{N} \right)
$$
```
                   THE SHANNON-HARTLEY CAPACITY BOUND
                   
    Capacity C ^
    [bits/sec] |                               Infinite Bandwidth Asymptote:
               |                         -----------------------------------
               |                      --'      C_inf = (S / N0) * (1 / ln 2)
               |                   .-'
               |                .-'
               |             .-'
               |          .-'
               |       .-'
               +----------------------------------------------------->
Bandwidth B
               0
               Notice the diminishing returns: doubling B does not double C
               at high bandwidth because total noise power N = N0 * B grows!
```
---
### The Infinite Bandwidth Limit
Can you achieve infinite communication capacity by making bandwidth
$B \to \infty$? No! As bandwidth increases, the receiver captures more noise:
$N = N_0 B \to \infty$.

Let us compute the limit as $B \to \infty$:
Recall the fundamental calculus limit: $\lim_{u \to 0} \frac{\ln(1 + u)}{u} =
1$.
Let $u = \frac{S}{N_0 B}$. As $B \to \infty$, $u \to 0$:
$$
\begin{aligned}
C_\infty &= \lim_{B \to \infty} B \log_2\left( 1 + \frac{S}{N_0 B} \right) \\
&= \lim_{B \to \infty} \frac{B}{\ln 2} \ln\left( 1 + \frac{S}{N_0 B} \right) \\
&= \lim_{B \to \infty} \frac{B}{\ln 2} \cdot \frac{S}{N_0 B} \cdot \frac{\ln(1
+ u)}{u} \\
&= \frac{S}{N_0 \ln 2} \approx 1.4427 \frac{S}{N_0} \quad [\text{bits/second}]
\end{aligned}
$$
> **The Ultimate Shannon Limit ($E_b / N_0$)**:
> Let $R$ be the transmission rate in bits/second, and let $E_b = S / R$ be the
> energy required per bit.
> For reliable communication ($R \le C$):
> $$
> R \le \frac{S}{N_0 \ln 2} = \frac{R \cdot E_b}{N_0 \ln 2}
> \implies \frac{E_b}{N_0} \ge \ln 2 \approx 0.693
> $$
> In decibels:
> $$
> \left( \frac{E_b}{N_0} \right)_{\text{min}} = 10 \log_{10}(\ln 2)
> \approx \mathbf{-1.59 \text{ dB}}
> $$
> No physical wireless or wired communication system in the universe can
transmit
> data error-free with less than $-1.59$ dB of energy per bit relative to the
> thermal noise floor!

---
## 5. Applied Engineering Contexts
### 1. Analog-to-Digital Converters (ADCs) and Quantization Noise
Every digital sensor—from audio interfaces to smartphone cameras and medical
imaging equipment—converts continuous analog voltages $V(t)$ into discrete
digital words of $b$ bits.

```
                    THE UNIFORM ADC QUANTIZATION PROCESS
                    
     Analog Input Voltage V                Discrete Quantized Level
     
           |                                     --- Level k+1
           |  <- Quantization Error e ->          
     ======o===================================  --- Level k
           |     Interval Delta = V_pp / 2^b
           |                                     --- Level k-1
```
Let the peak-to-peak input voltage range be $V_{\text{pp}}$. A $b$-bit converter
divides this range into $2^b$ uniform quantization bins of width:
$$
\Delta = \frac{V_{\text{pp}}}{2^b}
$$
The quantization error $e = X - X_{\text{quant}}$ is modeled as a uniform
random variable on $[-\Delta/2, \Delta/2]$. Its variance (noise power) is:
$$
\sigma_e^2 = \int_{-\Delta/2}^{\Delta/2} \frac{1}{\Delta} x^2 \, dx
= \frac{\Delta^2}{12} = \frac{V_{\text{pp}}^2}{12 \cdot 2^{2b}}
$$
For a full-scale sinusoidal input of amplitude $V_{\text{pp}}/2$, the signal
power is:
$$
P_{\text{signal}} = \frac{(V_{\text{pp}}/2)^2}{2} = \frac{V_{\text{pp}}^2}{8}
$$
Compute the Signal-to-Quantization-Noise Ratio (SQNR):
$$
\text{SQNR} = \frac{P_{\text{signal}}}{\sigma_e^2}
= \frac{V_{\text{pp}}^2 / 8}{V_{\text{pp}}^2 / (12 \cdot 2^{2b})}
= \frac{3}{2} \cdot 2^{2b}
$$
Expressed in decibels:
$$
\text{SQNR}_{\text{dB}} = 10 \log_{10}\left(\frac{3}{2}\right) + 10
\log_{10}(2^{2b})
= 1.76 + 20 b \log_{10}(2) \approx \mathbf{6.02 b + 1.76 \text{ dB}}
$$
#### The Information-Theoretic Interpretation
- Each additional bit ($b \to b + 1$) halves the quantization interval $\Delta$.
- By the coordinate scaling property, halving $\Delta$ decreases the
  quantization differential entropy by:
  $$
  \log_2(2) = 1.0 \text{ bit}
  $$
- In power terms, halving $\Delta$ quarters the noise variance $\sigma_e^2$,
  producing an immediate **6.02 dB increase** in signal quality.

---
### 2. Modern 5G/6G Wireless & Massive MIMO
In cellular communications, base stations and user handsets communicate over
multipath wireless channels. The channel between $M$ transmit antennas and $K$
receive antennas is modeled as:
$$
\mathbf{y} = \mathbf{H}\mathbf{x} + \mathbf{z}
$$
where $\mathbf{z} \sim \mathcal{N}_{\mathbb{C}}(\mathbf{0}, N_0 \mathbf{I})$ is
circularly symmetric complex Gaussian noise.

- By the Multivariate Gaussian MaxEnt Theorem, the worst-case noise that
  minimizes channel capacity is spatially white Gaussian noise.
- To maximize the mutual information $I(\mathbf{x}; \mathbf{y})$, the
  transmitter must shape the covariance matrix of $\mathbf{x}$ using the
  **Water-Filling Algorithm** across the singular values of channel matrix
  $\mathbf{H}$.
- High-gain spatial channels receive more transmit power, while subchannels
  below the noise floor receive zero power, maximizing total transmitted bits.

---
### 3. Thermal Johnson-Nyquist Noise in Physical Sensors
Any electrical resistor of resistance $R$ at temperature $T$ Kelvins
generates spontaneous, fluctuating voltage noise due to thermal agitation of
charge carriers (electrons).

In 1928, John B. Johnson measured this phenomenon, and Harry Nyquist proved its
information-theoretic foundation using statistical mechanics:
$$
S_v(f) = 4 k_B T R \quad [\text{Volts}^2 / \text{Hz}]
$$
where $k_B \approx 1.3806 \times 10^{-23}$ J/K is Boltzmann's constant.

- Why is thermal noise strictly Gaussian?
  Because macroscopic noise is the superposition of trillions of independent
  microscopic electron collisions. By the **Central Limit Theorem**, the sum
  of independent random variables converges to a Gaussian distribution.
- Why is thermal noise the worst possible noise?
  Because by the Gaussian Maximum Entropy Theorem, for a given thermal energy
  $k_B T$, Gaussian noise possesses the **maximum possible differential
entropy**.
  Thermal noise is nature's ultimate jammer!

---
## 6. Chapter Summary & What Lies Ahead
In this chapter, we crossed the boundary from discrete codes to continuous
measurements:
1. **The Discretization Paradox**: Discretizing continuous space yields
   divergent entropy $H(X^\Delta) \to \infty$. Subtracting the coordinate
   resolution $\log_2(1/\Delta)$ yields Claude Shannon's **Differential
Entropy**
   $h(X) = -\int f(x) \log_2 f(x) dx$.
2. **The Two Paradoxes**: Differential entropy can be negative ($h < 0$) and
   scales with coordinates ($h(cX) = h(X) + \log_2|c|$). In contrast,
   **KL divergence** and **Mutual Information** are strictly
coordinate-invariant.
3. **The Gaussian Theorem**: The normal distribution uniquely maximizes
   differential entropy among all distributions with fixed variance $\sigma^2$,
   achieving $h = \frac{1}{2}\log_2(2\pi e \sigma^2)$ bits.
4. **The Shannon-Hartley Law**: Capacity over an AWGN channel is bounded by
   $C = B \log_2(1 + S/N)$, with an ultimate physical limit of
   $E_b / N_0 \ge -1.59$ dB.

In **Chapter 7**, we ask the profound inverse question:
*If Gaussian distributions maximize entropy for a fixed variance, how do we
systematically find the maximum entropy distribution for ANY arbitrary set of
physical constraints?*

We will study Edwin T. Jaynes' **Principle of Maximum Entropy (MaxEnt)**—the
foundational philosophy of honest Bayesian inference and modern statistical
physics.

---
**[Course Home](../README.md) |
[Python Lab](../python/06_differential_entropy_and_continuous_limits.ipynb) |
[Prev: Ch 05](05_channel_capacity_and_blahut_arimoto.ipynb) |
[Next: Ch 07](07_the_principle_of_maximum_entropy_maxent.ipynb)**
